In [1]:
import sys
sys.path.append('../src/')

from trailo.data.datasets import SPLIT_SIZES, TrialGPTCriterionDataset

/home/ali/trialo/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = TrialGPTCriterionDataset("train")
validation = TrialGPTCriterionDataset("validation")
len(train), len(validation)

(603, 206)

In [3]:
sample = train[0]
sample

{'annotation_id': 43,
 'patient_id': 'sigir-20143',
 'note': '0. A 58-year-old nonsmoker white female with mild exertional dyspnea and occasional cough is found to have a left lung mass on chest x-ray.\n1. She is otherwise asymptomatic.\n2. A neurologic examination is unremarkable, but a CT scan of the head shows a solitary mass in the right frontal lobe.\n3. The patient will provide informed consent, and will comply with the trial protocol without any practical issues.',
 'trial_id': 'NCT01452971',
 'trial_title': 'A Study of the Interaction Between Tumor Susceptibility Gene Glycine N-methyltransferase (GNMT) and Lung Cancer',
 'criterion_type': 'inclusion',
 'criterion_text': 'Participant is a lung cancer patient.',
 'expert_eligibility': 'not enough information',
 'expert_sentences': [0]}

In [4]:
from torch.utils.data import DataLoader

loader = DataLoader(train, batch_size=2, collate_fn=lambda batch: batch)
batch = next(iter(loader))
[len(batch), batch[0]["annotation_id"], batch[1]["annotation_id"]]

[2, 43, 44]

In [5]:
import json
import time

import torch

from trailo.eval.inference import generate_prediction
from trailo.paths import CHECKPOINTS_DIR, repo_path
from trailo.train.modeling import load_adapter_model, load_model, load_tokenizer

MODEL_ID = "google/gemma-3n-E4B-it"
ADAPTER = CHECKPOINTS_DIR / "final_a100_seed42" / "best"
DEVICE = torch.device("cuda")

adapter_step = json.loads((ADAPTER / "metadata.json").read_text())["step"]
tokenizer = load_tokenizer(MODEL_ID, None)
base = load_model(MODEL_ID, None, DEVICE, "bf16", "sdpa", False)
model = load_adapter_model(base, str(ADAPTER))
model.to(DEVICE)
model.eval()
model.config.use_cache = True
print(f"base model + LoRA adapter loaded (best checkpoint at step {adapter_step})")

Loading weights:   0%|          | 0/1631 [00:00<?, ?it/s]

Loading weights:  26%|██▌       | 419/1631 [00:00<00:00, 4178.76it/s]

Loading weights:  51%|█████▏    | 837/1631 [00:00<00:00, 3827.94it/s]

Loading weights:  75%|███████▍  | 1222/1631 [00:00<00:00, 3833.25it/s]

Loading weights: 100%|██████████| 1631/1631 [00:00<00:00, 4008.52it/s]

base model + LoRA adapter loaded (best checkpoint at step 75)


In [6]:
label_sets = json.loads(
    (repo_path("data/processed/final_a100") / "label_sets.json").read_text()
)
records = [
    json.loads(line)
    for line
    in (repo_path("data/processed/final_a100") / "validation.jsonl")
    .read_text()
    .splitlines()
    if line
]
decoding = {
    "max_new_tokens": 128,
    "do_sample": False,
    "num_beams": 1,
    "label_sets": label_sets,
}
# small inference subset: first 10 validation examples
subset = records[:10]
len(subset)

10

In [7]:
rows = []
for index, record in enumerate(subset, start=1):
    started = time.time()
    output = generate_prediction(model, tokenizer, record, decoding)
    prediction = output["prediction"]
    gold = record["gold_label"]
    rows.append(
        {
            "annotation_id": record["annotation_id"],
            "criterion_type": record["criterion_type"],
            "gold": gold,
            "pred": prediction["label"],
            "evidence_pred": prediction["evidence_ids"],
            "evidence_gold": record["gold_evidence_ids"],
            "valid": output["valid"],
            "generated_tokens": output["generated_tokens"],
            "seconds": time.time() - started,
        }
    )
    mark = "OK " if prediction["label"] == gold else "MISS"
    print(
        f"[{index:2d}] {mark} gold={gold!r:26s} "
        f"pred={prediction['label']!r:26s} "
        f"evidence(pred)={prediction['evidence_ids']} "
        f"evidence(gold)={record['gold_evidence_ids']}"
    )

label_correct = sum(r["gold"] == r["pred"] for r in rows)
evidence_exact = sum(r["evidence_pred"] == r["evidence_gold"] for r in rows)
all_valid = sum(r["valid"] for r in rows)
print()
print(f"label exact match : {label_correct}/{len(rows)}")
print(f"evidence exact    : {evidence_exact}/{len(rows)}")
print(f"valid JSON parses : {all_valid}/{len(rows)}")

[ 1] OK  gold='not enough information'   pred='not enough information'   evidence(pred)=[0, 1] evidence(gold)=[0, 1, 2]


[ 2] OK  gold='included'                 pred='included'                 evidence(pred)=[0] evidence(gold)=[0]


[ 3] OK  gold='included'                 pred='included'                 evidence(pred)=[0] evidence(gold)=[0]


[ 4] OK  gold='included'                 pred='included'                 evidence(pred)=[8] evidence(gold)=[8]


[ 5] OK  gold='included'                 pred='included'                 evidence(pred)=[8] evidence(gold)=[8]


[ 6] OK  gold='not excluded'             pred='not excluded'             evidence(pred)=[3] evidence(gold)=[]


[ 7] OK  gold='not excluded'             pred='not excluded'             evidence(pred)=[] evidence(gold)=[]


[ 8] OK  gold='not excluded'             pred='not excluded'             evidence(pred)=[] evidence(gold)=[]


[ 9] MISS gold='not enough information'   pred='not excluded'             evidence(pred)=[0, 1] evidence(gold)=[0, 1, 2]


[10] OK  gold='not excluded'             pred='not excluded'             evidence(pred)=[] evidence(gold)=[]

label exact match : 9/10
evidence exact    : 7/10
valid JSON parses : 10/10
